# 02 - App Readiness Validation
Validates that all required tables/columns for dashboard sections exist.

In [0]:
from pyspark.sql import functions as F

CATALOG = "avant_users"
SCHEMA = "kaley_ubellacker"

BRONZE = f"{CATALOG}.{SCHEMA}.trustpilot_reviews_bronze"
SILVER = f"{CATALOG}.{SCHEMA}.trustpilot_reviews_silver"
GOLD = f"{CATALOG}.{SCHEMA}.trustpilot_sentiment_gold"

required = {
    BRONZE: ["review_id", "company", "rating", "text", "created_ts"],
    SILVER: ["review_id", "company", "rating", "text", "primary_category", "sentiment_score", "created_ts"],
    GOLD: ["company", "primary_category", "review_count", "avg_sentiment_score", "competitive_sentiment_index"],
}

for table, cols in required.items():
    exists = spark.catalog.tableExists(table)
    print(f"{table}: {'FOUND' if exists else 'MISSING'}")
    if not exists:
        continue
    actual = set(spark.table(table).columns)
    missing = [c for c in cols if c not in actual]
    if missing:
        print(f"  Missing columns: {missing}")
    else:
        print("  ✅ Required columns present")

# Optional derived views used by advanced sections
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.vw_monthly_sentiment_trend AS
SELECT company, date_trunc('month', created_ts) AS month_start, avg(sentiment_score) AS avg_sentiment
FROM {SILVER}
GROUP BY company, date_trunc('month', created_ts)
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.vw_category_heatmap AS
SELECT company, primary_category, avg(avg_sentiment_score) AS category_score
FROM {GOLD}
GROUP BY company, primary_category
""")

print("\n✅ Created/Refreshed supporting views:")
print(f"- {CATALOG}.{SCHEMA}.vw_monthly_sentiment_trend")
print(f"- {CATALOG}.{SCHEMA}.vw_category_heatmap")

avant_users.kaley_ubellacker.trustpilot_reviews_bronze: FOUND
  ✅ Required columns present
avant_users.kaley_ubellacker.trustpilot_reviews_silver: FOUND
  ✅ Required columns present
avant_users.kaley_ubellacker.trustpilot_sentiment_gold: FOUND
  ✅ Required columns present

✅ Created/Refreshed supporting views:
- avant_users.kaley_ubellacker.vw_monthly_sentiment_trend
- avant_users.kaley_ubellacker.vw_category_heatmap
